# A05 · Fine-tuning: SFT, LoRA & QLoRA

**Track:** Advanced / LLM engineering (lesson 5 of 10) · **Time:** ~2.5 h · **Mode:** runs fully OFFLINE (NumPy only, CPU, ~1 min); no API key needed
**Prereqs:** A01-A04 (APIs, prompting, RAG, evals), B02 (linear regression and gradient descent), B09 (backprop) · **You will build:** a decision framework, a chat-format loss masker, a LoRA layer with hand-derived gradients, a block quantizer (QLoRA idea), and a forgetting evaluation

---
## 🎯 Why this matters
Fine-tuning is the most over-prescribed and under-understood tool in the LLM toolbox. Teams reach for it to "teach the model our knowledge" (usually a RAG problem) or to "make it follow our format" (often a prompt or structured-output problem). Yet when it *is* the right tool - a stable behaviour, a strict style, a small cheap model that must punch above its weight - nothing else comes close.

**Real scenario.** *Kiwibank-style support triage:* a retail bank receives 40,000 customer messages a day. A large model with a long few-shot prompt classifies each into `{category, priority}` correctly, but the 1,800-token prompt makes every call slow and expensive. Should the team fine-tune a small model on 20,000 historical, human-labelled tickets? How? What could silently go wrong (bad labels, leakage, training on the prompt, forgetting)? And how does LoRA let them do it on a single GPU?

## 🧭 Learning objectives
By the end you can:
1. Apply a decision framework (prompt vs RAG vs fine-tune) to a concrete requirement and justify it with numbers (break-even volume).
2. Audit an SFT dataset for duplicates, label conflicts, eval leakage and PII, and build chat-format examples with correct **loss masking**.
3. Derive and implement **LoRA** `W' = W + (alpha/r) B A` from scratch, verify its gradients numerically, count parameters, and show that a merged adapter is exactly equivalent.
4. Run a **rank-vs-quality** experiment and explain the result from the rank of the true weight update.
5. Explain **QLoRA** (4-bit frozen base + LoRA adapters) with a block-quantizer error demo, and evaluate **forgetting**.

## 1 · Concept: three levers, three different problems

| Lever | Changes | Best when | Typical failure |
|---|---|---|---|
| **Prompting** (A02) | the *instructions* in context | behaviour is easy to describe; needs to change often | long prompts, inconsistent format |
| **RAG** (A03) | the *knowledge* in context | facts are private, large, or change daily; need citations | retrieval misses; stale index |
| **Fine-tuning** | the *weights* | stable skill/style/format; latency or cost pressure; small model must match a big one | stale knowledge baked in; forgetting; expensive to iterate |

A useful slogan: **RAG gives the model knowledge, fine-tuning gives it behaviour**. Fine-tuning on facts is a poor way to add them (they are memorised unreliably, cannot be cited, and cannot be deleted), while RAG cannot reliably change tone or a strict output grammar.

**Order of operations** (cheapest to most expensive to iterate on): write an eval (A04) -> better prompt -> few-shot examples -> structured outputs -> RAG -> fine-tune. Fine-tune only when you have a measured gap the cheaper levers cannot close, *and* a labelled dataset you trust.

Below we turn this into code, so the reasoning is explicit and testable.

In [ ]:
from dataclasses import dataclass

@dataclass
class Need:
    """A requirement, described by the properties that actually drive the choice."""
    knowledge_changes_often: bool = False     # facts update weekly/daily
    needs_private_facts: bool = False         # answer depends on documents the base model never saw
    needs_citations: bool = False
    behaviour_is_stable: bool = True          # the style/format/skill will not change next month
    easy_to_describe_in_prompt: bool = True   # a human can state the rule in a paragraph
    labelled_examples: int = 0                # trusted (input, ideal output) pairs
    calls_per_day: int = 0
    latency_critical: bool = False
    prompt_already_tried: bool = False        # a measured, serious prompt+few-shot baseline exists

def recommend(n: Need):
    """Return (list of levers, list of reasons). Deliberately simple and explainable."""
    levers, why = [], []
    if not n.prompt_already_tried:
        return ["prompt"], ["No measured prompt baseline yet: always start there (and write the eval first)."]
    if n.knowledge_changes_often or n.needs_private_facts or n.needs_citations:
        levers.append("rag"); why.append("Facts are private/changing or need citations: put them in context (RAG), not in weights.")
    if (not n.easy_to_describe_in_prompt or n.latency_critical or n.calls_per_day > 20000) and n.behaviour_is_stable:
        if n.labelled_examples >= 500:
            levers.append("fine-tune"); why.append("Stable behaviour that is hard to describe or costly to prompt, and >=500 trusted examples exist.")
        else:
            why.append("Fine-tuning would help but there are <500 trusted examples: collect/label data first (or use few-shot).")
    if not levers:
        levers.append("prompt"); why.append("Prompt + structured output is enough; keep iterating on the eval set.")
    return levers, why

cases = {
  "HR policy chatbot (policies edited weekly, must cite)": Need(knowledge_changes_often=True, needs_private_facts=True, needs_citations=True, prompt_already_tried=True),
  "Ticket triage, 40k calls/day, 1.8k-token prompt": Need(labelled_examples=20000, calls_per_day=40000, latency_critical=True, prompt_already_tried=True, easy_to_describe_in_prompt=False),
  "Weekly report summariser, tone changes with the CEO": Need(behaviour_is_stable=False, prompt_already_tried=True),
  "Brand-new idea, nothing tried": Need(),
  "Legal clause style, hard to describe, only 80 examples": Need(easy_to_describe_in_prompt=False, labelled_examples=80, prompt_already_tried=True),
}
for name, need in cases.items():
    levers, why = recommend(need)
    print(f"{name}\n   -> {levers}   {why[0]}")

assert recommend(cases["HR policy chatbot (policies edited weekly, must cite)"])[0] == ["rag"]
assert recommend(cases["Ticket triage, 40k calls/day, 1.8k-token prompt"])[0] == ["fine-tune"]
assert recommend(cases["Brand-new idea, nothing tried"])[0] == ["prompt"]
assert "fine-tune" not in recommend(cases["Legal clause style, hard to describe, only 80 examples"])[0]

### The economics: break-even volume
Fine-tuning has a fixed cost (data labelling, training, evaluation, maintenance) and *can* lower the per-call cost (shorter prompt, smaller model). The question is simply: **after how many calls does it pay back?**

Let `P` = prompt tokens in the baseline, `P_ft` = prompt tokens after fine-tuning, `c_in` = baseline cost per input token, `c_in_ft` = cost per input token of the fine-tuned (smaller) model, and `F` = one-off cost. Output tokens are the same in both. Saving per call `s = P*c_in - P_ft*c_in_ft`; break-even calls `= F / s` (undefined/never if `s <= 0`). **All numbers below are illustrative assumptions in "cost units", not real prices** - plug in your own from the provider's price list.

In [ ]:
def break_even_calls(P, P_ft, c_in, c_in_ft, fixed_cost):
    saving = P * c_in - P_ft * c_in_ft
    return float("inf") if saving <= 0 else fixed_cost / saving

# Assumptions (cost units): baseline model 1.0 unit per 1M input tokens; small fine-tuned model 0.2 units per 1M; one-off cost 400 units.
be = break_even_calls(P=1800, P_ft=120, c_in=1.0e-6, c_in_ft=0.2e-6, fixed_cost=400)
print(f"break-even after {be:,.0f} calls  =  {be/40_000:.1f} days at 40k calls/day")
assert 200_000 < be < 250_000          # 400 / (1.8e-3 - 2.4e-5) = 225,225
assert break_even_calls(100, 100, 1, 1, 10) == float("inf")   # no per-call saving -> never pays back
print("Insight: the saving comes from the shorter prompt AND the cheaper model; with a short prompt, fine-tuning may never pay back on cost alone (then you buy latency or quality instead).")

## 2 · SFT data: quality beats quantity, and the mask matters
**Supervised fine-tuning (SFT)** is next-token prediction on (prompt, ideal response) pairs. For a decoder, the loss is the cross-entropy *only on the response tokens*:

$$\mathcal{L}(\theta) = -\frac{1}{|\mathcal{R}|}\sum_{t \in \mathcal{R}} \log p_\theta(x_t \mid x_{<t})$$

where $\mathcal{R}$ is the set of positions we want the model to *produce* (assistant tokens). Everything else - system prompt, user message, and role markers - is **context**: it is fed in, but its positions get label `-100` (ignored). If you forget the mask you teach the model to *generate user messages*, dilute the gradient with tokens it should not learn, and (in multi-turn data) can teach it to hallucinate the next user turn.

A real tokenizer/chat template differs per model, so we use a toy whitespace tokenizer with explicit role tokens to make the mechanics visible.

In [ ]:
import numpy as np, re, json, random
IGNORE = -100
ROLE = {"system": "<|system|>", "user": "<|user|>", "assistant": "<|assistant|>"}
END = "<|end|>"

class ToyTokenizer:
    def __init__(self): self.vocab = {}
    def tok(self, text):
        return re.findall(r"<\|\w+\|>|\w+|[^\w\s]", text)
    def encode(self, toks):
        return [self.vocab.setdefault(t, len(self.vocab)) for t in toks]
tk = ToyTokenizer()

def build_example(messages, train_on="all_assistant"):
    """Return (tokens, labels). labels[t] = token id if position t is supervised else IGNORE.
    (Frameworks shift labels by one inside the loss; here labels are aligned with tokens for readability.)"""
    toks, sup = [], []
    for i, m in enumerate(messages):
        header = [ROLE[m["role"]]]
        body = tk.tok(m["content"]) + [END]
        is_target = m["role"] == "assistant" and (train_on == "all_assistant" or i == len(messages) - 1)
        toks += header + body
        sup += [False] + [is_target] * len(body)         # role marker is context; content + <|end|> are targets
    ids = tk.encode(toks)
    labels = [t if s else IGNORE for t, s in zip(ids, sup)]
    return ids, labels

demo = [
  {"role": "system", "content": "You triage bank support tickets. Reply with JSON only."},
  {"role": "user", "content": "My card was declined at the supermarket but I have money."},
  {"role": "assistant", "content": '{"category": "card_declined", "priority": "high"}'},
]
ids, labels = build_example(demo)
inv = {v: k for k, v in tk.vocab.items()}
for i, l in zip(ids, labels):
    print(f"{inv[i]:>16}   {'TRAIN' if l != IGNORE else '  -  '}")
n_sup = sum(l != IGNORE for l in labels)
n_expected = len(tk.tok(demo[2]["content"])) + 1      # assistant body + <|end|>
assert n_sup == n_expected, (n_sup, n_expected)
print(f"\nsupervised tokens: {n_sup} of {len(ids)}  ({n_sup/len(ids):.0%}) - the end token is supervised so the model learns WHEN TO STOP")

### Dataset audit
Most fine-tuning failures are data failures. Below is a synthetic 20k-ticket-style export (we generate 600 for speed) with realistic defects *seeded on purpose*: exact duplicates, the same ticket labelled two different ways by two annotators, rows copied from the held-out eval set (**leakage** - inflates your metric), and a customer card number left in the text (PII). A good audit finds every one of them, and the counts must match what we injected.

In [ ]:
rng = random.Random(7)
CATS = {
  "card_declined":   ["my card was declined at {place}", "payment refused at {place} but I have funds", "{place} would not accept my card"],
  "lost_card":       ["I lost my card near {place}", "my card was stolen at {place}", "cannot find my debit card since {place}"],
  "fee_dispute":     ["why was I charged a fee at {place}", "unexpected monthly fee, please explain, I was at {place} yesterday", "dispute an overdraft fee after {place}"],
  "app_login":       ["cannot log in to the app since the update, I was at {place}", "app crashes on login screen", "face id login fails on my phone"],
}
PRIORITY = {"card_declined": "high", "lost_card": "urgent", "fee_dispute": "normal", "app_login": "normal"}
PLACES = ["the supermarket", "Auckland airport", "a petrol station", "the cafe on Queen Street", "the pharmacy"]

def make_ticket(i):
    cat = rng.choice(list(CATS)); text = rng.choice(CATS[cat]).format(place=rng.choice(PLACES))
    return {"id": f"T{i:04d}", "text": f"{text} (ref {rng.randint(100000, 999999)})", "category": cat, "priority": PRIORITY[cat]}

train = [make_ticket(i) for i in range(500)]
eval_set = [make_ticket(1000 + i) for i in range(100)]
N_DUP, N_CONFLICT, N_LEAK, N_PII = 12, 7, 5, 4
for t in rng.sample(train[:300], N_DUP): train.append(dict(t, id=t["id"] + "dup"))
for t in rng.sample(train[:300], N_CONFLICT):
    other = rng.choice([c for c in CATS if c != t["category"]]); train.append(dict(t, id=t["id"] + "x", category=other, priority=PRIORITY[other]))
for t in rng.sample(eval_set, N_LEAK): train.append(dict(t, id=t["id"] + "leak"))
for k in range(N_PII):
    t = make_ticket(2000 + k); t["text"] += f" my card is 4{rng.randint(100,999)} {rng.randint(1000,9999)} {rng.randint(1000,9999)} {rng.randint(1000,9999)}"; train.append(t)
print(len(train), "training rows;", len(eval_set), "eval rows")

In [ ]:
from collections import defaultdict
norm = lambda s: re.sub(r"\s+", " ", re.sub(r"\(ref \d+\)", "", s.lower())).strip()   # the ref number is noise, not content
CARD = re.compile(r"\b(?:\d[ -]?){13,16}\b")

def audit(train, eval_set):
    seen, dup, by_text = {}, [], defaultdict(set)
    for r in train:
        key = (r["text"], r["category"])
        if key in seen: dup.append(r["id"])
        seen[key] = r["id"]
        by_text[r["text"]].add(r["category"])
    conflicts = [t for t, cats in by_text.items() if len(cats) > 1]
    eval_texts = {r["text"] for r in eval_set}
    leaks = [r["id"] for r in train if r["text"] in eval_texts]
    pii = [r["id"] for r in train if CARD.search(r["text"])]
    return {"exact_duplicates": len(dup), "label_conflicts": len(conflicts), "eval_leaks": len(leaks), "pii_rows": len(pii)}

report = audit(train, eval_set); print(report)
assert report["exact_duplicates"] == N_DUP
assert report["label_conflicts"] >= 1 and report["label_conflicts"] <= N_CONFLICT
assert report["eval_leaks"] == N_LEAK
assert report["pii_rows"] == N_PII
print("Every seeded defect found. (Conflicts <= injected: two injected rows can share the same random ticket text.)")

Why each defect matters:

| Defect | Consequence | Fix |
|---|---|---|
| duplicates | over-weight some examples; memorisation; fake eval gains if they straddle splits | dedupe on a normalised key *before* splitting |
| label conflicts | the model learns the *average* of annotators; irreducible loss; unpredictable edge behaviour | adjudicate, measure inter-annotator agreement, drop or relabel |
| eval leakage | your held-out metric is measuring memory | split by ticket/customer, hash-check every eval row against train |
| PII | the model can regurgitate it; compliance exposure | scrub *before* training; it is hard to remove afterwards |
| training on the prompt | wasted capacity; model learns to emit user turns | mask labels (above) |

A rule of thumb that holds up in practice: **a few hundred to a few thousand clean, diverse examples usually beats tens of thousands of noisy ones** for behaviour/format tuning. (It is a heuristic - measure it on *your* eval set by training on 10%, 30%, 100% of the data and plotting the learning curve.)

## 3 · Concept: LoRA - learn a low-rank *change*, freeze the rest
Full fine-tuning updates every weight matrix $W \in \mathbb{R}^{d_{out}\times d_{in}}$: that needs gradients and optimizer state (Adam keeps two extra tensors) for *all* parameters. The LoRA hypothesis (Hu et al., 2021) is that **the update $\Delta W$ needed to adapt a pretrained model to a task has low intrinsic rank**. So we freeze $W$ and learn

$$W' = W + \frac{\alpha}{r}\, B A,\qquad B\in\mathbb{R}^{d_{out}\times r},\; A\in\mathbb{R}^{r\times d_{in}},\; r \ll \min(d_{in}, d_{out})$$

* **Init:** $A \sim \mathcal{N}(0, 1/d_{in})$, $B = 0$ - so $\Delta W = 0$ at the start and the adapted model *is* the base model.
* **Scale** $s=\alpha/r$: keeps the update magnitude roughly stable as you change $r$, so one learning rate transfers across ranks.
* **Parameters:** $r(d_{in}+d_{out})$ instead of $d_{in}d_{out}$.

**Gradients.** Let $L$ be the loss and $G = \partial L/\partial W'$ (the gradient full fine-tuning would use). Since $W' = W + sBA$, by the chain rule

$$\frac{\partial L}{\partial B} = s\, G A^\top \in \mathbb{R}^{d_{out}\times r},\qquad \frac{\partial L}{\partial A} = s\, B^\top G \in \mathbb{R}^{r\times d_{in}}.$$

Note the consequence of $B=0$ at init: $\partial L/\partial A = 0$ on the first step, so only $B$ moves first; then $A$ starts to learn. **At inference** you can *merge*: $W_{merged} = W + sBA$ - zero extra latency - or keep adapters separate and swap them per customer/task.

Parameter-count arithmetic first (pure arithmetic, no training needed):

In [ ]:
def lora_params(d_in, d_out, r): return r * (d_in + d_out)

# A single 4096x4096 attention projection (a typical size for a ~7B-class model; illustrative)
d = 4096
full = d * d
print(f"full matrix: {full:,} params")
for r in [1, 4, 8, 16, 64]:
    p = lora_params(d, d, r)
    print(f"  r={r:<3} LoRA: {p:>9,} params = {p/full:.3%} of the matrix")
assert lora_params(d, d, 8) == 65_536 and abs(lora_params(d, d, 8) / full - 2 * 8 / d) < 1e-12   # = 2r/d for square matrices

# Whole-model illustration: 32 layers x 4 attention projections (q,k,v,o), all 4096x4096, rank 16
n_layers, n_mats, r = 32, 4, 16
trainable = n_layers * n_mats * lora_params(d, d, r)
print(f"\nrank-16 LoRA on q,k,v,o in 32 layers: {trainable/1e6:.1f}M trainable params; a 7B-parameter base is {7e9/trainable:,.0f}x larger")
# Optimizer-state memory (fp32 Adam keeps 2 extra values per trainable param, plus the gradient)
print(f"Adam state+grad for these: {trainable*4*3/1e6:.0f} MB   vs   for all 7B params: {7e9*4*3/1e9:.0f} GB")

## 4 · Build it: LoRA on a layer we can verify
To *see* the mechanism with certainty we use a setting where the ground truth is known. A "pretrained" linear layer $W_0$ must adapt to a new task whose true weights are

$$W_1 = W_0 + \underbrace{\Delta_{lr}}_{\text{rank } 4} + \underbrace{\Delta_{res}}_{\text{small, full rank}}$$

i.e. mostly a rank-4 change plus a small full-rank remainder (real adaptations are *approximately* low-rank, not exactly). We have 200 noisy labelled examples. Questions: does LoRA with $r\ge4$ match full fine-tuning? What does $r<4$ cost? The model is linear, $\hat y = W x$, with loss $L=\frac{1}{2n}\sum_i\|\hat y_i-y_i\|^2$, so $G=\frac1n(\hat Y-Y)^\top X$.

In [ ]:
d_in, d_out, r_true = 64, 48, 4
g = np.random.default_rng(0)
W0 = g.normal(size=(d_out, d_in)) / np.sqrt(d_in)
scale_to = lambda M, f: M * f / np.linalg.norm(M)
D_lr  = scale_to(g.normal(size=(d_out, r_true)) @ g.normal(size=(r_true, d_in)), 4.0)   # rank-4 change, Frobenius norm 4
D_res = scale_to(g.normal(size=(d_out, d_in)), 0.8)                                       # full-rank remainder, norm 0.8
W1 = W0 + D_lr + D_res
assert np.linalg.matrix_rank(D_lr) == r_true and np.linalg.matrix_rank(W1 - W0) == min(d_in, d_out)

spec = (np.arange(d_in) + 1.0) ** -0.5; spec /= np.sqrt((spec ** 2).mean())   # task-B inputs: decaying feature spectrum (narrow domain)
sampleB = lambda n: g.normal(size=(n, d_in)) * spec
sampleA = lambda n: g.normal(size=(n, d_in))                                   # "general" inputs
Xtr = sampleB(200); Ytr = Xtr @ W1.T + 0.1 * g.normal(size=(200, d_out))
XB, XA = sampleB(4000), sampleA(4000)
YB, YA = XB @ W1.T, XA @ W0.T          # test targets: new task B (noise-free), and "old behaviour" A = what the base model did
mse = lambda W, X, Y: float(((X @ W.T - Y) ** 2).mean())
print(f"base model: task-B MSE {mse(W0, XB, YB):.3f} (needs adapting);  task-A MSE {mse(W0, XA, YA):.3f} (by definition)")

First the layer and its hand-derived gradients, **checked against finite differences** before we trust any training curve. (A gradient bug in a custom layer never crashes; it just trains worse.)

In [ ]:
class LoRALinear:
    def __init__(self, W, r, alpha=None, seed=1):
        self.W, self.r = W, r
        self.alpha = alpha if alpha is not None else 2 * r
        self.s = self.alpha / r
        gg = np.random.default_rng(seed)
        self.A = gg.normal(size=(r, W.shape[1])) / np.sqrt(W.shape[1])    # ~ N(0, 1/d_in)
        self.B = np.zeros((W.shape[0], r))                                # zero init -> Delta W = 0 at start
    def weight(self): return self.W + self.s * self.B @ self.A            # merged weight (what inference could use)
    def forward(self, X): return X @ self.W.T + self.s * (X @ self.A.T) @ self.B.T   # adapter path, never forms BA
    def n_trainable(self): return self.A.size + self.B.size
    def grads(self, X, Y):
        Gw = ((self.forward(X) - Y).T @ X) / len(X)                       # G = dL/dW'
        return {"B": self.s * Gw @ self.A.T, "A": self.s * self.B.T @ Gw}

def loss_fn(layer, X, Y): return 0.5 * float(((layer.forward(X) - Y) ** 2).sum(axis=1).mean())

# gradient check at a point where B != 0 (otherwise dL/dA is trivially 0)
chk = LoRALinear(W0, r=3, alpha=6, seed=5); chk.B = np.random.default_rng(9).normal(size=chk.B.shape) * 0.1
Xs, Ys = Xtr[:20], Ytr[:20]
an = chk.grads(Xs, Ys)
for name in ["A", "B"]:
    P = getattr(chk, name); num = np.zeros_like(P); eps = 1e-6
    for idx in np.ndindex(*P.shape):
        old = P[idx]; P[idx] = old + eps; lp = loss_fn(chk, Xs, Ys); P[idx] = old - eps; lm = loss_fn(chk, Xs, Ys); P[idx] = old
        num[idx] = (lp - lm) / (2 * eps)
    rel = np.abs(num - an[name]).max() / (np.abs(num).max() + 1e-12)
    print(f"grad check {name}: max relative error {rel:.2e}")
    assert rel < 1e-6
z = LoRALinear(W0, r=3); assert np.allclose(z.grads(Xs, Ys)["A"], 0), "with B=0 the gradient into A must vanish on step 1"
assert np.allclose(z.forward(Xs), Xs @ W0.T), "at init the adapted model IS the base model"
print("OK: gradients verified; adapter starts as an exact no-op.")

Now an optimizer (Adam, written out) and a trainer for both full fine-tuning and LoRA. We train each for the same number of steps on the same 200 examples.

In [ ]:
class Adam:
    def __init__(self, lr, b1=0.9, b2=0.999, eps=1e-8): self.lr, self.b1, self.b2, self.eps, self.t, self.m, self.v = lr, b1, b2, eps, 0, {}, {}
    def step(self, params, grads):
        self.t += 1
        for k, gk in grads.items():
            self.m[k] = self.b1 * self.m.get(k, 0) + (1 - self.b1) * gk
            self.v[k] = self.b2 * self.v.get(k, 0) + (1 - self.b2) * gk ** 2
            mh, vh = self.m[k] / (1 - self.b1 ** self.t), self.v[k] / (1 - self.b2 ** self.t)
            params[k] -= self.lr * mh / (np.sqrt(vh) + self.eps)

def train_full(X, Y, steps=800, lr=3e-3):
    p = {"W": W0.copy()}; opt = Adam(lr)
    for _ in range(steps): opt.step(p, {"W": ((X @ p["W"].T - Y).T @ X) / len(X)})
    return p["W"]

def train_lora(X, Y, r, steps=800, lr=1e-2, alpha=None, seed=1, extra=None):
    layer = LoRALinear(W0, r, alpha, seed); opt = Adam(lr); params = {"A": layer.A, "B": layer.B}   # Adam updates arrays in place
    for _ in range(steps): opt.step(params, layer.grads(X, Y))
    return layer

W_full = train_full(Xtr, Ytr)
print(f"full FT : task-B MSE {mse(W_full, XB, YB):.4f}   task-A drift {mse(W_full, XA, YA):.4f}   trainable {W0.size:,}")

### The rank-vs-quality experiment
Sweep $r \in \{1,2,4,8,16,32\}$. Prediction from the theory: the true update is rank 4 plus a small full-rank remainder, so test error should fall steeply until $r=4$, then flatten at (almost) the full-FT level - with a tiny fraction of the parameters. Below rank 4 the adapter *cannot represent* the update, whatever the optimizer does.

In [ ]:
import matplotlib.pyplot as plt
ranks = [1, 2, 4, 8, 16, 32]
res = {}
for r in ranks:
    L = train_lora(Xtr, Ytr, r); Wm = L.weight()
    res[r] = {"B": mse(Wm, XB, YB), "A": mse(Wm, XA, YA), "params": L.n_trainable()}
    print(f"r={r:<3} task-B MSE {res[r]['B']:.4f}   task-A drift {res[r]['A']:.4f}   trainable {res[r]['params']:>6,} ({res[r]['params']/W0.size:.0%} of full)")

# reference: truncated SVD of the TRUE update (Eckart-Young best rank-r approximation in Frobenius norm)
U, S, Vt = np.linalg.svd(W1 - W0)
floor = {r: mse(W0 + (U[:, :r] * S[:r]) @ Vt[:r], XB, YB) for r in ranks}

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].semilogy(ranks, [res[r]["B"] for r in ranks], "o-", label="LoRA (trained)")
ax[0].semilogy(ranks, [floor[r] for r in ranks], "s--", label="truncated-SVD update (reference)")
ax[0].axhline(mse(W_full, XB, YB), color="k", ls=":", label="full fine-tune"); ax[0].axvline(r_true, color="grey", lw=0.8)
ax[0].set_xscale("log", base=2); ax[0].set_xlabel("LoRA rank r"); ax[0].set_ylabel("task-B test MSE"); ax[0].set_title("quality vs rank"); ax[0].legend()
ax[1].plot(S[:12], "o-"); ax[1].set_xlabel("index"); ax[1].set_ylabel("singular value of true update"); ax[1].set_title("spectrum of W1 - W0: a knee at 4")
plt.tight_layout(); plt.show()

assert res[1]["B"] > res[2]["B"] > res[4]["B"]                  # monotone gain until the true rank
assert res[4]["B"] < 0.1 * res[2]["B"]                          # the knee: rank 4 captures the dominant update
assert res[8]["B"] < res[4]["B"] * 1.0 and res[32]["B"] < 1.5 * mse(W_full, XB, YB) + 1e-3   # beyond the knee: diminishing returns
assert res[4]["params"] / W0.size < 0.4
assert floor[4] < 0.1 * floor[2] and floor[1] > floor[2]       # the truncated-SVD reference shows the same knee (it is not a strict bound under anisotropic inputs)

**Reading the plot.** The singular values of the true update drop sharply after index 4, and trained-LoRA error follows the truncated-SVD reference. That is the LoRA hypothesis made concrete: *if* the needed change is low-rank, a small adapter recovers nearly all the gain (here 15% of the layer's parameters at r=4; for a real 4096x4096 matrix the same rank is 0.2%, because the saving factor is $d_{in}d_{out}/(r(d_{in}+d_{out}))$ and grows with $d$ - note that in our tiny 48x64 layer r=32 is *larger* than the full layer, so the saving only exists when $r \ll d$). For real LLM fine-tuning the "knee" is softer and task-dependent (harder tasks, e.g. learning a new programming language, need higher rank than a style change) - which is why you sweep $r$ on your own eval rather than trusting a default.

### Merge check
The merged weight must reproduce the adapter model **exactly** (up to floating point). This is also how you would verify a merge in a real stack before shipping the merged checkpoint.

In [ ]:
L8 = train_lora(Xtr, Ytr, r=8)
Xchk = sampleB(500)
out_adapter, out_merged = L8.forward(Xchk), Xchk @ L8.weight().T
print("max |adapter - merged| =", np.abs(out_adapter - out_merged).max())
assert np.allclose(out_adapter, out_merged, atol=1e-10)
# swapping adapters = swapping (A,B); removing the adapter restores the base model bit-for-bit
assert np.array_equal(Xchk @ W0.T, Xchk @ L8.W.T), "base weights were never modified"
print("merge verified; base W untouched (adapter files are tiny: ", L8.n_trainable() * 4 / 1e3, "KB in fp32 here)")

## 5 · Use it: QLoRA - quantize the frozen base, keep adapters in high precision
The frozen base weights are by far the biggest memory consumer, and they are *never updated*, so they can be stored in low precision. **QLoRA** (Dettmers et al., 2023) stores the base in 4-bit, de-quantizes block-by-block on the fly in the forward pass, and trains LoRA adapters (kept in higher precision) on top. The gradient flows *through* the dequantized weights to the adapters; the base itself never changes.

**Block-wise absmax quantization.** Split weights into blocks of $b$; for each block store one scale $s=\max|w|$ and integers $q=\mathrm{round}(w/s\cdot q_{max})$ with $q_{max}=2^{k-1}-1$ for $k$ bits. Smaller blocks => scales adapt to local outliers => lower error, at the price of more scale metadata ($+16$ or $32$ bits per block). Real 4-bit formats additionally choose their 16 levels non-uniformly to match the roughly-normal distribution of weights (the "NormalFloat" idea); we implement a quantile-based codebook in that spirit to compare - it is *not* the exact published NF4 table.

In [ ]:
def quant_absmax(w, bits, block):
    w = w.reshape(-1, block); s = np.abs(w).max(axis=1, keepdims=True) + 1e-12; qmax = 2 ** (bits - 1) - 1
    q = np.round(w / s * qmax).astype(np.int8); return q, s, qmax
def dequant_absmax(q, s, qmax): return q.astype(np.float64) / qmax * s

def quant_codebook(w, block, levels):
    """Non-uniform: snap each (block-normalised) weight to the nearest of 16 codebook values."""
    w = w.reshape(-1, block); s = np.abs(w).max(axis=1, keepdims=True) + 1e-12; x = w / s
    idx = np.abs(x[..., None] - levels[None, None, :]).argmin(axis=-1)
    return levels[idx] * s

# codebook = quantiles of a standard normal, rescaled to [-1,1] (sampled quantiles, so it adapts to bell-shaped weights)
samp = np.sort(np.random.default_rng(0).normal(size=200_000))
levels = samp[((np.arange(16) + 0.5) / 16 * len(samp)).astype(int)]; levels = levels / np.abs(levels).max()

gw = np.random.default_rng(3)
w = gw.normal(size=1 << 16) * 0.02                                     # bell-shaped weights...
w[gw.choice(w.size, 40, replace=False)] *= 12                          # ...with a few outliers (they exist in real LLMs)
print(f"{'scheme':<28}{'block':>6}{'RMSE':>12}{'bits/weight (+fp16 scale)':>28}")
rows = {}
for bits in [8, 4]:
    for block in [4096, 256, 64, 16]:
        q, s, qm = quant_absmax(w, bits, block); err = np.sqrt(((dequant_absmax(q, s, qm).ravel() - w) ** 2).mean())
        rows[(bits, block)] = err; print(f"{'int%d absmax' % bits:<28}{block:>6}{err:>12.6f}{bits + 16/block:>28.2f}")
for block in [256, 64]:
    err = np.sqrt(((quant_codebook(w, block, levels).ravel() - w) ** 2).mean()); rows[("cb", block)] = err
    print(f"{'4-bit quantile codebook':<28}{block:>6}{err:>12.6f}{4 + 16/block:>28.2f}")

assert rows[(8, 64)] < rows[(4, 64)] / 5            # 4 more bits cut error by well over 5x
assert rows[(4, 16)] < rows[(4, 256)] < rows[(4, 4096)]   # smaller blocks: outliers hurt fewer neighbours
assert rows[("cb", 64)] < rows[(4, 64)]            # matching levels to the weight distribution helps at the same bit-width

Now the QLoRA logic end-to-end on our layer: quantize the *base* $W_0$ to 4 bits (so the model is slightly "wrong" even before adaptation), then train a rank-8 LoRA on top of the quantized base. A well-known effect is that the adapter can partly **compensate for quantization error** on the task distribution, because the adapter is trained with the quantized weights in the loop.

In [ ]:
def fake_quant_matrix(W, bits=4, block=16):
    q, s, qm = quant_absmax(W.copy().ravel(), bits, block); return dequant_absmax(q, s, qm).reshape(W.shape)

W0_q = fake_quant_matrix(W0, 4, 16)
print(f"relative weight error of 4-bit base: {np.linalg.norm(W0_q - W0)/np.linalg.norm(W0):.3f}")

def train_lora_on(Wbase, X, Y, r, steps=800, lr=1e-2):
    layer = LoRALinear(Wbase, r); opt = Adam(lr); params = {"A": layer.A, "B": layer.B}
    for _ in range(steps): opt.step(params, layer.grads(X, Y))
    return layer

qlora = train_lora_on(W0_q, Xtr, Ytr, r=8)
lora16 = train_lora_on(W0, Xtr, Ytr, r=8)
table = {"base (fp)": mse(W0, XB, YB), "base quantized (no adapter)": mse(W0_q, XB, YB),
         "LoRA r=8 on fp base": mse(lora16.weight(), XB, YB), "QLoRA r=8 on 4-bit base": mse(qlora.weight(), XB, YB)}
for k, v in table.items(): print(f"{k:<30} task-B MSE {v:.4f}")
assert table["QLoRA r=8 on 4-bit base"] < 0.05 * table["base quantized (no adapter)"]
# memory for the base weights alone, 7B parameters (arithmetic, not a benchmark)
for name, bits in [("fp16", 16), ("int8", 8), ("4-bit (+scale/block 64)", 4 + 16 / 64)]:
    print(f"7B base weights in {name:<24}: {7e9 * bits / 8 / 1e9:5.1f} GB")

## 6 · Use it: does adaptation hurt what the model could already do?
**Catastrophic forgetting**: after fine-tuning on a narrow domain, performance on *other* behaviours degrades. Never ship a fine-tune without a **retention eval** - a fixed set of general-capability checks run on base and tuned models, with a regression budget you decide in advance (e.g. "no more than 2% relative drop").

In our linear toy, "task A" is just *what the base model already computed* on general inputs; its error is $\|(W-W_0)\Sigma_A^{1/2}\|_F^2/d_{out}$, i.e. how far the weights moved along directions the general data excites. Three things to compare: full FT, LoRA at several ranks, and LoRA **plus replay** (mix 25% general-data examples, labelled by the base model, into the training batch - the standard cheap mitigation).

In [ ]:
def train_lora_replay(r, replay_frac=0.25, steps=800, lr=1e-2):
    n_rep = int(len(Xtr) * replay_frac / (1 - replay_frac)); Xr = sampleA(n_rep); Yr = Xr @ W0.T      # replay: base model's own outputs
    return train_lora_on(W0, np.vstack([Xtr, Xr]), np.vstack([Ytr, Yr]), r, steps, lr)

rows = [("base", W0), ("full FT", W_full)]
rows += [(f"LoRA r={r}", train_lora_on(W0, Xtr, Ytr, r).weight()) for r in [1, 4, 16]]
rows += [("LoRA r=16 + 25% replay", train_lora_replay(16).weight())]
print(f"{'model':<26}{'new-task MSE':>14}{'retention drift':>18}")
ret = {}
for name, W in rows:
    ret[name] = (mse(W, XB, YB), mse(W, XA, YA)); print(f"{name:<26}{ret[name][0]:>14.4f}{ret[name][1]:>18.4f}")

assert ret["base"][1] == 0.0
assert ret["LoRA r=1"][1] < ret["LoRA r=16"][1] < ret["full FT"][1] * 1.05      # smaller update -> less drift (with a tolerance)
assert ret["LoRA r=16 + 25% replay"][1] < ret["LoRA r=16"][1]                  # replay buys retention...
assert ret["LoRA r=16 + 25% replay"][0] > ret["LoRA r=16"][0] * 0.9            # ...and costs a bit of new-task fit (no free lunch)
print("\nTrade-off visible: LoRA rank and replay move you along a learn-more / forget-more frontier. Pick the point your retention budget allows.")

This matches the direction reported in the literature ("LoRA learns less and forgets less", Biderman et al., 2024), but be careful about over-generalising from a *linear toy with isotropic general data*: in real LLMs forgetting depends on data overlap, learning rate, number of epochs and which layers are adapted. The engineering takeaways are robust, though: (1) measure retention, (2) the smaller the weight change, the smaller the worst-case drift, (3) removing an adapter returns the *exact* base model - a rollback story full fine-tuning does not have.

## 7 · Break it
### 7a · Training on the prompt (forgotten mask)
What fraction of the gradient signal goes to the wrong place if you supervise everything? In multi-turn chat data with a long system prompt it is most of it:

In [ ]:
long_sys = "You triage bank support tickets for a New Zealand retail bank. " + "Follow the policy strictly. " * 30
conv = [{"role": "system", "content": long_sys}, {"role": "user", "content": "card declined at the pharmacy"},
        {"role": "assistant", "content": '{"category": "card_declined", "priority": "high"}'}]
ids, lab = build_example(conv)
frac_masked = sum(l != IGNORE for l in lab) / len(lab)
print(f"with correct masking only {frac_masked:.0%} of positions carry loss; without it {1.0:.0%} do -> {1/frac_masked:.0f}x more loss mass on text you never want generated")
assert frac_masked < 0.2
ids2, lab2 = build_example(conv + [{"role": "user", "content": "thanks"}, {"role": "assistant", "content": "You are welcome."}], train_on="last_assistant")
assert sum(l != IGNORE for l in lab2) == len(tk.tok("You are welcome.")) + 1, "train_on='last_assistant' supervises only the final reply"

### 7b · Too much capacity, too little data
With 200 examples and a 64x48 layer, full fine-tuning and high-rank LoRA also fit the *label noise*. Shrink the training set to 60 examples and compare test error across ranks - the best rank moves down.

In [ ]:
Xs_ = sampleB(60); Ys_ = Xs_ @ W1.T + 0.3 * np.random.default_rng(4).normal(size=(60, d_out))      # tiny, noisier set
small = {r: mse(train_lora_on(W0, Xs_, Ys_, r).weight(), XB, YB) for r in [1, 2, 4, 8, 16, 32]}
small["full"] = mse(train_full(Xs_, Ys_), XB, YB)
print({k: round(v, 3) for k, v in small.items()})
best = min(small, key=small.get); print("best setting on 60 noisy examples:", best)
assert small["full"] > small[4], "full fine-tuning overfits the noise more than a rank-4 adapter when data is scarce"

### 7c · The `alpha/r` scale and the learning rate
The purpose of dividing by $r$ (Hu et al., 2021) is to reduce the need to re-tune the learning rate when you change $r$: with `alpha` fixed, the update's scale stays comparable across ranks, whereas a scale of $\alpha$ (no $1/r$) lets the update grow with the number of rank-1 terms. Measure how far $\Delta W$ moves in the first 20 Adam steps. **Caveat, measured honestly:** Adam normalises gradient magnitudes, so in this toy the effect is modest; it is larger with plain SGD and in real training, where it interacts with warmup and learning-rate schedules.

In [ ]:
def delta_norm_after(r, scale_mode, steps=20, lr=1e-2, alpha=16):
    layer = LoRALinear(W0, r, alpha=alpha, seed=1); layer.s = alpha / r if scale_mode == "alpha/r" else alpha
    opt = Adam(lr); params = {"A": layer.A, "B": layer.B}
    for _ in range(steps): opt.step(params, layer.grads(Xtr, Ytr))
    return np.linalg.norm(layer.s * layer.B @ layer.A)
for mode in ["alpha/r", "alpha"]:
    print(mode.ljust(8), {r: round(float(delta_norm_after(r, mode)), 2) for r in [2, 8, 32]})
a = [delta_norm_after(r, "alpha/r") for r in [2, 32]]; b = [delta_norm_after(r, "alpha") for r in [2, 32]]
assert b[1] / b[0] > a[1] / a[0], "without the 1/r factor the update grows (somewhat) faster with rank"
print(f"growth from r=2 to r=32:  alpha/r x{a[1]/a[0]:.2f}   alpha x{b[1]/b[0]:.2f}")
print("Keep alpha/r scaling and tune lr per task; (some practitioners prefer alpha/sqrt(r) - an empirical choice, test it on your eval.)")

## 8 · Exercises
**Practice (30 min)**
1. A 13B-parameter-class model has 40 layers and you adapt `q,k,v,o` (each 5120x5120) with rank 16. Compute the number of trainable parameters and the adapter file size in fp16. Write the code and assert your arithmetic against `lora_params`.
2. Extend `audit` to flag *near-duplicates* (same normalised text via `norm`, ignoring the ref number) and report how many extra rows it finds.

**Stretch (60 min)**
3. Repeat the rank sweep with a true update of rank 8 and then rank 16. Plot the knee location against the true rank and assert it moves.
4. Add *weight decay on the adapter* to `train_lora_on` and see whether it improves the 60-example overfitting experiment (7b).

**Challenge (2 h)**
5. Implement **LoRA+ style** different learning rates for `A` and `B` (e.g. `lr_B = 8 * lr_A`) and measure steps-to-reach a target test MSE against equal learning rates. Report a table over 3 seeds. (Only claim what you measure.)

In [ ]:
# Practice 1 starter - make the assertions pass
def adapter_size(n_layers, n_mats, d, r, bytes_per_param=2):
    # TODO: return (n_trainable_params, file_size_bytes)
    raise NotImplementedError

# n, size = adapter_size(40, 4, 5120, 16)
# assert n == 40 * 4 * lora_params(5120, 5120, 16)
# assert size == 2 * n
# print(f"{n/1e6:.1f}M params, {size/1e6:.1f} MB")

## ✅ Checkpoint
1. A colleague wants to fine-tune a model "so it knows our product catalogue, which changes every Friday." What do you recommend and why?
2. In `W' = W + (alpha/r) B A`, why is `B` initialised to zero and `A` random (rather than both random, or both zero)?
3. Your LoRA rank sweep shows test loss flat from r=8 to r=64. What does that suggest about the task's update, and what do you choose?
4. Quantizing the base model to 4 bits, why can the model still be fine-tuned effectively, and what is *not* quantized?

<details><summary>Answers</summary>

1. RAG (A03), not fine-tuning: the facts change weekly, need to be citable and deletable, and weights are a poor, stale store for facts. Fine-tune only if there is *also* a stable behaviour problem (e.g. strict output style) that prompts cannot fix.
2. With $B=0$ the adapter is an exact no-op at step 0, so training starts from the base model's behaviour. With $A$ random the gradient into $B$ ($sGA^\top$) is non-zero, so learning begins immediately; both zero would give zero gradients to both (dead start), and both random would perturb the base model before any learning.
3. The needed update is effectively low-rank (rank <= ~8); extra rank only adds parameters, overfitting risk and forgetting. Choose the smallest rank at the knee (about 8, maybe lower - test it) and confirm on a retention eval.
4. The base is frozen, so quantization error is a fixed (if slightly harmful) distortion that the adapters learn *around*; gradients are computed through the dequantized weights. The LoRA adapters (and optimizer state) stay in higher precision (e.g. bf16/fp16/fp32), as do activations.
</details>

## 📌 Takeaways
- **Prompt -> RAG -> fine-tune**, in that order of cost. Fine-tune for *behaviour* with a measured gap; RAG for *knowledge*.
- SFT quality is data quality: dedupe, adjudicate conflicts, split before you look, scrub PII, and **mask the loss to assistant tokens**.
- LoRA = frozen $W$ + low-rank $\frac{\alpha}{r}BA$: $r(d_{in}+d_{out})$ parameters, exact merge, zero-init no-op, gradients $sGA^\top$ and $sB^\top G$ (verified numerically).
- Rank is a capacity knob tied to the rank of the *needed update*; sweep it, don't guess.
- QLoRA = 4-bit frozen base + high-precision adapters; block size and level placement control quantization error.
- Always run a **retention eval** next to the task eval; adapters are also a rollback mechanism.

## 📚 Further reading
- Hu et al., *LoRA: Low-Rank Adaptation of Large Language Models* (2021), arXiv:2106.09685
- Dettmers et al., *QLoRA: Efficient Finetuning of Quantized LLMs* (2023), arXiv:2305.14314
- Biderman et al., *LoRA Learns Less and Forgets Less* (2024)
- Ouyang et al., *Training language models to follow instructions with human feedback* (2022), arXiv:2203.02155 (SFT as the first stage of the pipeline)

**Next -> A06 · Preference Tuning & RL for LLMs** (SFT teaches *what* to say; preferences teach *which of several acceptable answers is better*)